# S6E4 — 03 Adversarial validation (synthetic vs original data)

> Teammate's notebook from the team's Kaggle pipeline, cleaned for the repository: empty cells, `!ls` calls, duplicates and commented-out code removed; the code that ran is unchanged.

| | |
|---|---|
| **Input** | `train.csv`, `test.csv`, original 10k-row dataset |
| **Result** | AUC **0.688**: a classifier separates the synthetic rows from the original ones, so the original data is not a drop-in extension of train |

# Adversarial validation
It is a machine learning technique used to determine if training and test datasets follow the same distribution, primarily to detect and fix data drift or covariate shift

## Process
- Combine train and test sets, removing the original target variables.
- Label the training set 0 and the test set 1.
- Train a binary classifier (like XGBoost) to predict this new label.
- **Analyze the results**:  If the model can easily tell them apart (AUC > 0.5), it indicates the training data is not representative of the test data

## Interpretation of Results
**AUC ~ 0.5:** The datasets are indistinguishable; the model cannot tell them apart.  
**AUC > 0.5:** The datasets differ, often leading to overconfident training and poor test performance.

In [1]:
# Importing packages
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
import xgboost as xgb
from sklearn.metrics import roc_auc_score
import joblib

pd.set_option("display.max_columns", None)

In [2]:
train=pd.read_csv("/kaggle/input/competitions/playground-series-s6e4/train.csv")
test=pd.read_csv("/kaggle/input/competitions/playground-series-s6e4/test.csv")
original=pd.read_csv("/kaggle/input/datasets/miadul/irrigation-water-requirement-prediction-dataset/irrigation_prediction.csv")
print(train.shape, test.shape, original.shape)
train.head(2)

(630000, 21) (270000, 20) (10000, 20)


,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,16.79,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,3.39,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low


In [6]:
NUM_FEAT = [
    'Soil_pH','Soil_Moisture','Organic_Carbon','Electrical_Conductivity',
    'Temperature_C','Humidity','Rainfall_mm','Sunlight_Hours',
    'Wind_Speed_kmh','Field_Area_hectare','Previous_Irrigation_mm',
]

CAT_FEAT = [
    'Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season',
'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region', 
]

COMMON_COLUMNS = [i for i in original.columns if i in train.columns]
NUM_FEAT = [i for i in NUM_FEAT if i in COMMON_COLUMNS]
CAT_FEAT = [i for i in CAT_FEAT if i in COMMON_COLUMNS]

print("Total features:", len(NUM_FEAT+ CAT_FEAT))

Total features: 19


In [7]:
# Label training + test set as 1 and original as 0
train['istrain'] = 1
test['istrain'] = 1
original['istrain'] = 0

REQ_COLUMNS = NUM_FEAT+ CAT_FEAT + ['istrain']

xdat = pd.concat([train[REQ_COLUMNS], test[REQ_COLUMNS], original[REQ_COLUMNS]], axis = 0)

y = xdat['istrain']
xdat = xdat[NUM_FEAT+ CAT_FEAT]

for feat in CAT_FEAT:
    xdat[feat] = xdat[feat].astype('category') # Convert object dtype to category, so it can be processed by XGBoost automatically

In [8]:
# We perform K-Fold CV
skf = StratifiedKFold(n_splits = 5, shuffle = True, random_state = 42)
xgb_params = {
    'learning_rate': 0.05, 'max_depth': 4,'subsample': 0.8,
    'colsample_bytree': 0.8,'objective': 'binary:logistic',
    'n_estimators':100, 'gamma':1,
    'min_child_weight':4, 
    'enable_categorical': True, 'tree_method':"hist",
}   
clf = xgb.XGBClassifier(**xgb_params, seed = 42, eval_metric='logloss', early_stopping_rounds=10)

In [9]:
# Model training
auc_scores = []
for fold, (train_index, test_index) in enumerate(skf.split(xdat, y)):
    x0, x1 = xdat.iloc[train_index], xdat.iloc[test_index]
    y0, y1 = y.iloc[train_index], y.iloc[test_index]        
    
    clf.fit(x0, y0, eval_set=[(x1, y1)], verbose=False)
            
    prval = clf.predict_proba(x1)[:,1]
    score = roc_auc_score(y1,prval)
    print(f"Fold {fold} | AUC: {round(score, 6)}")
    auc_scores.append(score)

print(f"\nMean score: {round(sum(auc_scores)/len(auc_scores), 6)}")

Fold 0 | AUC: 0.683156
Fold 1 | AUC: 0.69384
Fold 2 | AUC: 0.686609
Fold 3 | AUC: 0.680105
Fold 4 | AUC: 0.694961

Mean score: 0.687734


### Mean AUC = 0.68 (baseline = 0.5), highlighting a strong signal of data drift.